# 00 -- Sanity checks for Meek's rules

Scratch space for checking the Meek implementation against examples whose
answers are known by hand. Not a test suite -- `tests/test_meek.py` is the
suite. This is for looking at a graph when a test fails and you need to see
what the closure actually did.

**Clear all outputs before committing.** The `nbstripout` pre-commit hook does
this, but only if it was installed (`pre-commit install`).


In [ ]:
from bkrobust.graphs.mpdag import MPDAG
from bkrobust.graphs import meek
from bkrobust.knowledge.base import BackgroundKnowledge

## The chain, and why it is the whole problem

`A -> B -> C` and `C -> B -> A` share the CPDAG `A - B - C`. The data cannot
separate them. An expert asserting the reverse of the truth passes every check
a practitioner runs and is wrong about both edges.


In [ ]:
chain = MPDAG(nodes=["A", "B", "C"], directed=[("A", "B"), ("B", "C")])
cpdag = MPDAG(nodes=["A", "B", "C"], undirected=[("A", "B"), ("B", "C")])

## R1: no new v-structures

Orienting `A -> B` on the chain's CPDAG forces `B -> C`, since otherwise
`A -> B <- C` would be a v-structure the data did not show.


In [ ]:
bk = BackgroundKnowledge().with_edge("A", "B")
closed = meek.apply_background_knowledge(cpdag, bk)
closed

## The cascade

One asserted orientation, more than one oriented edge. `forced_orientations`
returns only the ones the rules produced, not the ones asserted -- the
amplification factor is meaningless otherwise.


In [ ]:
meek.forced_orientations(cpdag, bk)

## FAIL

Knowledge that cannot be imposed returns `None`, not an exception. FAIL is an
expected outcome that the rejection sampler hits constantly.


In [ ]:
collider = MPDAG(nodes=["A", "B", "C"], directed=[("A", "B"), ("C", "B")])
bad = BackgroundKnowledge().with_edge("B", "A")
meek.apply_background_knowledge(collider, bad)  # expect None